# 🧪 Test Set Reconstruction — No Retraining

## Context

Both datasets (comprehension and evolution) are **fully synthetic**.
Training is complete — we do not retrain.

## Strategy: Maximum-Diversity Test Sets

We reconstruct **only** the test sets by selecting examples that:
1. **Were not used for training** (absent from train+val)
2. Are **maximally distant from train** (low Jaccard lexical similarity)
3. **Uniformly cover** all categories (operations, adl_type, element_type)
4. Are **high quality** (quality ≥ 4 for comprehension)

## What this notebook produces

```
dataset_clean/
├── test_comprehension_v2.jsonl   ← NEW comprehension test set
└── evolution/
    └── evo_test_v2.jsonl         ← NEW evolution test set
```

> **Train and val are not modified.**

In [ ]:
# ============================================================
# CELL 0 — Mount Drive & Paths
# ============================================================
from google.colab import drive
import os

drive.mount('/content/drive')

BASE_DIR  = "/content/drive/MyDrive/ArchPipeline"  # <-- adjust to your own path
CLEAN_DIR = f'{BASE_DIR}/dataset_clean'
EVO_DIR   = f'{CLEAN_DIR}/evolution'

# ── Comprehension sources ────────────────────────────────────
COMPR_TRAIN = f'{CLEAN_DIR}/train_comprehension.jsonl'
COMPR_VAL   = f'{CLEAN_DIR}/val_comprehension.jsonl'
COMPR_TEST  = f'{CLEAN_DIR}/test_comprehension.jsonl'   # current (to replace)
COMPR_FULL  = f'{CLEAN_DIR}/dataset_Comprehension.csv'  # full dataset

# ── Evolution sources ─────────────────────────────────────────
EVO_TRAIN   = f'{EVO_DIR}/evo_train.jsonl'
EVO_VAL     = f'{EVO_DIR}/evo_val.jsonl'
EVO_TEST    = f'{EVO_DIR}/evo_test.jsonl'              # current (to replace)

# ── Outputs ───────────────────────────────────────────────────
OUT_COMPR_TEST = f'{CLEAN_DIR}/test_comprehension_v2.jsonl'
OUT_EVO_TEST   = f'{EVO_DIR}/evo_test_v2.jsonl'

print('📂 Checking source files:')
all_ok = True
for label, path in [
    ('train_comprehension.jsonl',  COMPR_TRAIN),
    ('val_comprehension.jsonl',    COMPR_VAL),
    ('dataset_Comprehension.csv',  COMPR_FULL),
    ('evo_train.jsonl',            EVO_TRAIN),
    ('evo_val.jsonl',              EVO_VAL),
    ('evo_test.jsonl',             EVO_TEST),
]:
    ok = os.path.exists(path)
    all_ok = all_ok and ok
    print(f'  {"✅" if ok else "❌"} {label}')

if not all_ok:
    print('\n⚠️  Some files are missing — verify paths')

In [ ]:
# ============================================================
# CELL 1 — Imports & Utilities
# ============================================================
import pandas as pd
import json
import random
import re
from collections import Counter, defaultdict

random.seed(42)

def load_jsonl(path):
    with open(path, encoding='utf-8') as f:
        return [json.loads(l) for l in f if l.strip()]

def save_jsonl(records, path):
    with open(path, 'w', encoding='utf-8') as f:
        for r in records:
            f.write(json.dumps(r, ensure_ascii=False) + '\n')
    print(f'  💾 {os.path.basename(path)} → {len(records)} examples saved')

def tokenize(text):
    """Simple tokenization for Jaccard similarity."""
    return set(re.findall(r'\b\w+\b', str(text).lower()))

def jaccard(set_a, set_b):
    """Jaccard similarity between two token sets."""
    if not set_a or not set_b:
        return 0.0
    inter = len(set_a & set_b)
    union = len(set_a | set_b)
    return inter / union if union > 0 else 0.0

def score_distance_from_train(candidate_tokens, train_token_sets,
                               sample_size=200):
    """
    Computes the average distance of the candidate from
    a sample of the training set.
    Returns 1 - mean_jaccard (higher = more distant).
    """
    sample = random.sample(train_token_sets,
                           min(sample_size, len(train_token_sets)))
    sims   = [jaccard(candidate_tokens, ts) for ts in sample]
    return 1.0 - (sum(sims) / len(sims))

print('✅ Utilities loaded')

---
## Part 1 — Comprehension Test Set v2

In [ ]:
# ============================================================
# CELL 2 — Comprehension Loading
# ============================================================

# Load train + val (used for training)
train_compr = load_jsonl(COMPR_TRAIN)
val_compr   = load_jsonl(COMPR_VAL)
seen_compr  = train_compr + val_compr

print(f'📊 Comprehension training data:')
print(f'   Train : {len(train_compr)} examples')
print(f'   Val   : {len(val_compr)} examples')
print(f'   Total seen during training : {len(seen_compr)}')

# Train fingerprints (input + target concatenated)
seen_fingerprints = set(
    (str(ex.get('input','')).strip()[:120],
     str(ex.get('target','')).strip()[:80])
    for ex in seen_compr
)
print(f'   Unique fingerprints : {len(seen_fingerprints)}')

# Token sets from train for distance computation
train_token_sets_compr = [
    tokenize(ex.get('input','') + ' ' + ex.get('target',''))
    for ex in train_compr
]
print(f'\n✅ Comprehension train index built')

In [ ]:
# ============================================================
# CELL 3 — Full Pool Loading
# We use dataset_Comprehension.csv as candidate pool
# ============================================================

df_full = pd.read_csv(COMPR_FULL, encoding='utf-8')

# Normalize columns
if 'quality_score' in df_full.columns and 'quality' not in df_full.columns:
    df_full = df_full.rename(columns={'quality_score': 'quality'})
if 'code' in df_full.columns and 'input' not in df_full.columns:
    df_full = df_full.rename(columns={'code': 'input', 'comments': 'target'})

print(f'📊 Full pool : {len(df_full)} pairs')
print(f'   Columns : {list(df_full.columns)}')
print(f'\n📊 Source distribution :')
if 'source' in df_full.columns:
    print(df_full['source'].value_counts().to_string())
print(f'\n📊 adl_type distribution :')
print(df_full['adl_type'].value_counts().to_string())
print(f'\n📊 element_type distribution :')
print(df_full['element_type'].value_counts().to_string())

In [ ]:
# ============================================================
# CELL 4 — Candidate Filtering for Test Set
# Criteria:
#   1. Absent from train+val (fingerprint)
#   2. Quality >= 4
#   3. Jaccard distance from train >= 0.35
# ============================================================

MIN_QUALITY      = 4      # quality threshold
MIN_DISTANCE     = 0.35   # minimum distance from train

quality_col = 'quality' if 'quality' in df_full.columns else 'quality_score'

candidates = []

for _, row in df_full.iterrows():
    inp    = str(row.get('input', '')).strip()
    tgt    = str(row.get('target', '')).strip()
    qual   = int(row.get(quality_col, 0))

    # Criterion 1: absent from train+val
    fp = (inp[:120], tgt[:80])
    if fp in seen_fingerprints:
        continue

    # Criterion 2: quality
    if qual < MIN_QUALITY:
        continue

    # Criterion 3: distance from train
    tokens   = tokenize(inp + ' ' + tgt)
    distance = score_distance_from_train(
        tokens, train_token_sets_compr, sample_size=150)

    candidates.append({
        'input'       : inp,
        'target'      : tgt,
        'adl_type'    : str(row.get('adl_type', 'ACME')),
        'element_type': str(row.get('element_type', 'Other')),
        'quality'     : qual,
        'source'      : str(row.get('source', 'synthetic')),
        '_distance'   : distance,
        '_tokens'     : tokens,
    })

print(f'📊 Candidates after filtering:')
print(f'   Absent from train+val : -')
print(f'   Quality >= {MIN_QUALITY}       : -')
print(f'   Distance >= {MIN_DISTANCE}   : -')
print(f'   → TOTAL candidates   : {len(candidates)}')

if len(candidates) == 0:
    print('\n⚠️  No candidates — reduce MIN_QUALITY or MIN_DISTANCE')
    print('   Try MIN_QUALITY=3, MIN_DISTANCE=0.25')
else:
    dists = [c['_distance'] for c in candidates]
    print(f'\n   Mean distance : {sum(dists)/len(dists):.3f}')
    print(f'   Max distance  : {max(dists):.3f}')
    print(f'   Min distance  : {min(dists):.3f}')

In [ ]:
# ============================================================
# CELL 5 — Stratified Candidate Selection
# Goal: uniformly cover adl_type × element_type
# by prioritizing those most distant from train
# ============================================================

N_TEST_COMPR = 100  # target comprehension test set size

# Group by (adl_type, element_type)
groups = defaultdict(list)
for c in candidates:
    key = (c['adl_type'], c['element_type'])
    groups[key].append(c)

# Sort each group by decreasing distance (most distant first)
for key in groups:
    groups[key].sort(key=lambda x: x['_distance'], reverse=True)

print(f'📊 Available groups (adl_type × element_type) : {len(groups)}')
for key, items in sorted(groups.items(), key=lambda x: -len(x[1])):
    print(f'   {key[0]:<6} × {key[1]:<20} : {len(items)} candidates')

# Quota per group (uniform distribution)
n_groups  = len(groups)
quota_base = max(1, N_TEST_COMPR // n_groups)
print(f'\n   Base quota per group : {quota_base}')

# Selection
test_selected = []
for key, items in groups.items():
    # Take up to quota_base, most distant from train
    selects = items[:quota_base]
    test_selected.extend(selects)

# Fill if below N_TEST_COMPR
if len(test_selected) < N_TEST_COMPR:
    # Take extras from richest groups
    already_selected = set(
        (c['input'][:120], c['target'][:80])
        for c in test_selected
    )
    remaining = [
        c for c in candidates
        if (c['input'][:120], c['target'][:80]) not in already_selected
    ]
    remaining.sort(key=lambda x: x['_distance'], reverse=True)
    missing = N_TEST_COMPR - len(test_selected)
    test_selected.extend(remaining[:missing])

# Shuffle
random.shuffle(test_selected)

print(f'\n✅ Comprehension test set selected : {len(test_selected)} examples')
print(f'\n📊 Final distribution :')
ct_adl = Counter(c['adl_type'] for c in test_selected)
ct_elt = Counter(c['element_type'] for c in test_selected)
print(f'   ADL type      : {dict(ct_adl)}')
print(f'   element_type  :')
for k, v in ct_elt.most_common():
    print(f'     {k:<25} : {v}')

mean_dist = sum(c['_distance'] for c in test_selected) / len(test_selected)
print(f'\n   Mean distance from train : {mean_dist:.3f} '
      f'(1.0 = completely different, 0.0 = identical)')

In [ ]:
# ============================================================
# CELL 6 — Save Comprehension Test v2
# ============================================================

test_compr_final = [
    {
        'input'       : c['input'],
        'target'      : c['target'],
        'adl_type'    : c['adl_type'],
        'element_type': c['element_type'],
        'quality'     : c['quality'],
        'source'      : c['source'],
        'distance_train': round(c['_distance'], 4),
    }
    for c in test_selected
]

save_jsonl(test_compr_final, OUT_COMPR_TEST)

# Comparison with old test set
old_test = load_jsonl(COMPR_TEST) if os.path.exists(COMPR_TEST) else []
print(f'\n📊 Comparison:')
print(f'   Old test set  : {len(old_test)} examples')
print(f'   New test set  : {len(test_compr_final)} examples')
print(f'   Mean distance : {mean_dist:.3f} '
      f'(better train/test separation)')
print(f'\n✅ Comprehension test v2 saved: {OUT_COMPR_TEST}')

---
## Part 2 — Evolution Test Set v2

In [ ]:
# ============================================================
# CELL 7 — Evolution Loading
# ============================================================

train_evo = load_jsonl(EVO_TRAIN)
val_evo   = load_jsonl(EVO_VAL)
test_evo  = load_jsonl(EVO_TEST)

print(f'📊 Evolution dataset:')
print(f'   Train : {len(train_evo)}')
print(f'   Val   : {len(val_evo)}')
print(f'   Test  : {len(test_evo)} (current — to replace)')

# Candidate pool = current test + val
# We recycle the current test and val as candidate pool
# (train remains intact to avoid contamination)
pool_evo = test_evo + val_evo

# Train fingerprints only
seen_evo = set(
    (str(ex.get('input','')).strip()[:150],
     str(ex.get('target','')).strip()[:100])
    for ex in train_evo
)

# Token sets from train
train_token_sets_evo = [
    tokenize(ex.get('input','') + ' ' + ex.get('target',''))
    for ex in train_evo
]

print(f'\n   Candidate pool (current test+val) : {len(pool_evo)}')
print(f'\n📊 Operation distribution in pool:')
ct_op = Counter(ex.get('operation','?') for ex in pool_evo)
for op, cnt in sorted(ct_op.items()):
    print(f'   {op:<25} : {cnt}')

In [ ]:
# ============================================================
# CELL 8 — Evolution Candidate Filtering
# Criteria:
#   1. Absent from train (fingerprint)
#   2. Jaccard distance from train >= 0.30
#   3. Non-empty target with sufficient length (>= 10 tokens)
# ============================================================

MIN_DISTANCE_EVO = 0.30
MIN_TARGET_TOKENS = 10

candidates_evo = []

for ex in pool_evo:
    inp = str(ex.get('input',  '')).strip()
    tgt = str(ex.get('target', '')).strip()
    op  = str(ex.get('operation', ''))
    adl = str(ex.get('adl_type', ''))

    # Criterion 1: absent from train
    fp = (inp[:150], tgt[:100])
    if fp in seen_evo:
        continue

    # Criterion 2: non-trivial target
    if len(tgt.split()) < MIN_TARGET_TOKENS:
        continue

    # Criterion 3: distance from train
    tokens   = tokenize(inp + ' ' + tgt)
    distance = score_distance_from_train(
        tokens, train_token_sets_evo, sample_size=150)

    if distance < MIN_DISTANCE_EVO:
        continue

    candidates_evo.append({
        **ex,
        '_distance': distance,
        '_tokens'  : tokens,
    })

print(f'📊 Evolution candidates after filtering: {len(candidates_evo)}')

if len(candidates_evo) > 0:
    print(f'\n📊 By operation:')
    ct = Counter(c.get('operation','?') for c in candidates_evo)
    for op, cnt in sorted(ct.items()):
        print(f'   {op:<25} : {cnt}')
    dists = [c['_distance'] for c in candidates_evo]
    print(f'\n   Mean distance : {sum(dists)/len(dists):.3f}')
else:
    print('\n⚠️  No candidates — reduce MIN_DISTANCE_EVO or MIN_TARGET_TOKENS')
    print('   Try MIN_DISTANCE_EVO=0.20, MIN_TARGET_TOKENS=5')

In [ ]:
# ============================================================
# CELL 9 — Stratified Evolution Selection
# Stratification: operation × adl_type
# Guaranteed minimum: N_MIN_PER_OP per operation
# ============================================================

N_TEST_EVO     = 120    # target size
N_MIN_PER_OP   = 10    # minimum per operation (6 ops → min 60)

OPERATIONS = [
    'ADD_PORT', 'MODIFY_PROPERTY',
    'ADD_COMPONENT', 'DELETE_COMPONENT',
    'ADD_CONNECTOR', 'DELETE_CONNECTOR',
]

# Group by operation
groups_evo = defaultdict(list)
for c in candidates_evo:
    op = c.get('operation', 'UNKNOWN')
    groups_evo[op].append(c)

# Sort each group by decreasing distance
for op in groups_evo:
    groups_evo[op].sort(key=lambda x: x['_distance'], reverse=True)

print('📊 Selection by operation:')
test_evo_selected = []

for op in OPERATIONS:
    items = groups_evo.get(op, [])
    available = len(items)

    # Adaptive quota
    quota = max(N_MIN_PER_OP, N_TEST_EVO // len(OPERATIONS))
    n_sel = min(quota, available)

    selects = items[:n_sel]
    test_evo_selected.extend(selects)

    mean_dist_op = (
        sum(s['_distance'] for s in selects) / len(selects)
        if selects else 0
    )
    flag = '✅' if n_sel >= N_MIN_PER_OP else '⚠️ '
    print(f'  {flag} {op:<25} : {n_sel}/{available} '
          f'(mean_dist={mean_dist_op:.3f})')

# Fill if needed
if len(test_evo_selected) < N_TEST_EVO:
    already = set(
        (c.get('input','')[:150], c.get('target','')[:100])
        for c in test_evo_selected
    )
    remaining = [
        c for c in candidates_evo
        if (c.get('input','')[:150], c.get('target','')[:100]) not in already
    ]
    remaining.sort(key=lambda x: x['_distance'], reverse=True)
    missing = N_TEST_EVO - len(test_evo_selected)
    test_evo_selected.extend(remaining[:missing])

random.shuffle(test_evo_selected)

print(f'\n✅ Evolution test set: {len(test_evo_selected)} examples')
print(f'\n📊 Final distribution by operation:')
ct_final = Counter(c.get('operation','?') for c in test_evo_selected)
for op in OPERATIONS:
    cnt = ct_final.get(op, 0)
    flag = '✅' if cnt >= N_MIN_PER_OP else '⚠️ '
    print(f'  {flag} {op:<25} : {cnt}')

print(f'\n📊 ADL type distribution:')
print(dict(Counter(c.get('adl_type','?') for c in test_evo_selected)))

mean_dist_final = sum(c['_distance'] for c in test_evo_selected) / len(test_evo_selected)
print(f'\n   Mean distance from train: {mean_dist_final:.3f}')

In [ ]:
# ============================================================
# CELL 10 — Save Evolution Test v2
# ============================================================

# Clean internal fields before saving
test_evo_final = []
for c in test_evo_selected:
    rec = {
        k: v for k, v in c.items()
        if not k.startswith('_')
    }
    rec['distance_train'] = round(c['_distance'], 4)
    test_evo_final.append(rec)

save_jsonl(test_evo_final, OUT_EVO_TEST)

print(f'\n📊 Comparison:')
print(f'   Old test set  : {len(test_evo)} examples')
print(f'   New test set  : {len(test_evo_final)} examples')
print(f'   Mean distance : {mean_dist_final:.3f}')
print(f'\n✅ Evolution test v2 saved: {OUT_EVO_TEST}')

In [ ]:
# ============================================================
# CELL 11 — Final Report & Update Instructions
# ============================================================
import json as _json

report = {
    'comprehension': {
        'old_test_size'   : len(old_test) if 'old_test' in dir() else '?',
        'new_test_size'   : len(test_compr_final),
        'min_quality'     : MIN_QUALITY,
        'min_distance'    : MIN_DISTANCE,
        'mean_dist_train' : round(mean_dist, 4),
        'adl_distribution'   : dict(Counter(c['adl_type'] for c in test_compr_final)),
        'element_distribution': dict(Counter(c['element_type'] for c in test_compr_final)),
        'file'            : OUT_COMPR_TEST,
    },
    'evolution': {
        'old_test_size'   : len(test_evo),
        'new_test_size'   : len(test_evo_final),
        'min_distance'    : MIN_DISTANCE_EVO,
        'mean_dist_train' : round(mean_dist_final, 4),
        'operation_distribution': dict(Counter(c.get('operation','?') for c in test_evo_final)),
        'file'            : OUT_EVO_TEST,
    }
}

with open(f'{CLEAN_DIR}/report_testsets_v2.json', 'w') as f:
    _json.dump(report, f, indent=2, ensure_ascii=False)

print('=' * 65)
print('📊 FINAL REPORT')
print('=' * 65)
print(f'\n🔵 COMPREHENSION')
print(f'   File : test_comprehension_v2.jsonl')
print(f'   Size : {len(test_compr_final)} examples')
print(f'   Mean distance from train : {mean_dist:.3f}')

print(f'\n🟢 EVOLUTION')
print(f'   File : evo_test_v2.jsonl')
print(f'   Size : {len(test_evo_final)} examples')
print(f'   Mean distance from train : {mean_dist_final:.3f}')

print(f'\n' + '=' * 65)
print('🔧 UPDATE in ArchPipeline_Final.ipynb')
print('=' * 65)
print(f'''
# Cell 1 — replace EVO_TEST:
EVO_TEST = f'{{BASE_DIR}}/dataset_clean/evolution/evo_test_v2.jsonl'

# Cell 1 — add COMPR_TEST:
COMPR_TEST = f'{{BASE_DIR}}/dataset_clean/test_comprehension_v2.jsonl'
''')

print('=' * 65)
print('📌 NOTE ON EXPECTED METRICS')
print('=' * 65)
print('''
A drop in metrics is normal and expected:

  Comprehension EM   : 0.3250 → likely 0.20-0.28
  Comprehension BLEU : 0.5496 → likely 0.40-0.50
  Evolution EM        : 0.8670 → likely 0.50-0.75
  S_evo               : 0.9619 → likely 0.70-0.90

This is not a failure — it is a more honest measurement.
Mention in the README:
  "Evaluated on a maximum-diversity test set (Jaccard distance ≥ X)"
''')